In [1]:
import parcels
import numpy as np
from datetime import timedelta
from glob import glob
import matplotlib.pyplot as plt
import xarray as xr
import os

def u2rho_2d (var_u):
    [Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[:,1:-1]=0.5*(var_u[:,1:]+var_u[:,:-1])
    var_rho[:,0]=var_rho[:,1]
    var_rho[:,-1]=var_rho[:,-2]
    return var_rho
    
def v2rho_2d (var_v):
    [M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[1:-1,:]=0.5*(var_v[1:,:]+var_v[:-1,:])
    var_rho[0,:]=var_rho[1,:]
    var_rho[-1,:]=var_rho[-2,:]
    return var_rho

def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:-1]=0.5*(var_u[:,:,1:]+var_u[:,:,:-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,-1]=var_rho[:,:,-2]
    return var_rho
    
def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:-1,:]=0.5*(var_v[:,1:,:]+var_v[:,:-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,-1,:]=var_rho[:,-2,:]
    return var_rho

def spheric_dist(lat1, lat2, lon1, lon2):
    """
    Compute the spherical distance between two points on Earth.
    
    Parameters:
    lat1, lat2 : array-like
        Latitude of the two points (in degrees).
    lon1, lon2 : array-like
        Longitude of the two points (in degrees).
    
    Returns:
    dist : array-like
        The spherical distance between the points (in meters).
    """
    
    # Earth radius in meters
    R = 6367442.76
    
    # Determine proper longitudinal shift
    l = np.abs(lon2 - lon1)
    l[l >= 180] = 360 - l[l >= 180]
    
    # Convert decimal degrees to radians
    deg2rad = np.pi / 180
    lat1 = lat1 * deg2rad
    lat2 = lat2 * deg2rad
    l = l * deg2rad
    
    # Compute the distances
    dist = R * np.arcsin(np.sqrt(((np.sin(l) * np.cos(lat2)) ** 2) + 
                                 ((np.sin(lat2) * np.cos(lat1)) - 
                                  (np.sin(lat1) * np.cos(lat2) * np.cos(l))) ** 2))
    
    return dist

def spheric_dist_one(lat1, lat2, lon1, lon2):
    """计算两个经纬度点之间的球面距离（标量版）"""
    # 处理经度差
    l = np.abs(lon2 - lon1)
    if l >= 180:
        l = 360 - l
    
    # 转换为弧度
    deg2rad = np.pi / 180
    lat1_rad = lat1 * deg2rad
    lat2_rad = lat2 * deg2rad
    l_rad = l * deg2rad
    
    # 球面距离公式
    distance = 6371 * np.arccos(
        np.sin(lat1_rad) * np.sin(lat2_rad) + 
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.cos(l_rad)
    )
    return distance
def transunit_spher2flat(u,v,lat):
    v1=v*1852*60
    u1=u*1852*60*np.cos(lat*np.pi/180)
    return u1,v1

def trans_vel_roms(u,v,angle):
    # np.cos(angle*np.pi/180)
    # np.sin(angle*np.pi/180)
    u_east=u*np.cos(angle*np.pi/180)-v*np.sin(angle*np.pi/180)
    v_north=u*np.sin(angle*np.pi/180)+v*np.cos(angle*np.pi/180)
    return u_east,v_north

# Helmholtz

In [11]:
grid=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/niskin2km_500m_grd.nc')
angle=np.tile(grid['angle'].values[np.newaxis,:,:],[2148,1,1])
angle.shape

(2148, 287, 287)

# HF

In [13]:
ds1=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_wave/z_niskin2km_his_hf_depth_500m_grd.0002.nc')
utot=ds1['u'].values
vtot=ds1['v'].values
ds2=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_wave/helmholtz.0002.nc')
udiv=ds2['udiv'].values
vdiv=ds2['vdiv'].values
del ds1
del ds2
ds3=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_wave/rothelmholtz.0002.nc')
# ds3
urot=utot-udiv[:,np.newaxis,:,:]
vrot=vtot-vdiv[:,np.newaxis,:,:]

In [14]:
urot.shape

(2148, 1, 287, 286)

In [15]:
u1=u2rho_3d(np.squeeze(urot))
v1=v2rho_3d(np.squeeze(vrot))
ueast,vnorth=trans_vel_roms(u1,v1,angle)
ds3['u_rho'].values[:,0,:,:]=ueast
ds3['v_rho'].values[:,0,:,:]=vnorth
ds3['u'].values=urot
ds3['v'].values=vrot
ds3=ds3.drop_vars(['w','divof','Ro'])


In [16]:
ds3

<xarray.Dataset> Size: 4GB
Dimensions:     (depth: 1, eta_rho: 287, xi_rho: 287, time: 2148, xi_u: 286,
                 eta_v: 286)
Coordinates:
  * depth       (depth) float32 4B -2.0
    lat_rho     (eta_rho, xi_rho) float64 659kB ...
    lon_rho     (eta_rho, xi_rho) float64 659kB ...
  * time        (time) float64 17kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Dimensions without coordinates: eta_rho, xi_rho, xi_u, eta_v
Data variables:
    ocean_time  (time) float32 9kB ...
    u           (time, depth, eta_rho, xi_u) float32 705MB 0.06592 ... -0.03341
    u_rho       (time, depth, eta_rho, xi_rho) float64 1GB 0.08779 ... 0.002642
    v           (time, depth, eta_v, xi_rho) float32 705MB 0.09002 ... 0.2073
    v_rho       (time, depth, eta_rho, xi_rho) float64 1GB 0.07172 ... 0.2208
Attributes:
    CDI:          Climate Data Interface version 2.4.1 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon Oct 13 03:24:41 2025: ncks -x -v Th* wavecase_modified_...
    CDO:          Climate Data Operators version 2.4.1 (https://mpimet.mpg.de...
    NCO:          netCDF Operators version 5.3.5 (Homepage = http://nco.sf.ne...

In [17]:
ds3.to_netcdf('/meddy/simingzhang/Data/RB_iceland_data/iceland_wave/rot_helmholtz.0002.nc')
del ds3

# Smooth

In [20]:
ds1=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/z_niskin2km_his_smooth_depth_500m_grd.0002.nc')
utot=ds1['u'].values
vtot=ds1['v'].values
ds2=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/helmholtz.0002.nc')
udiv=ds2['udiv'].values
vdiv=ds2['vdiv'].values
del ds1
del ds2
ds3=xr.open_dataset('/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/rothelmholtz.0002.nc')
# ds3
urot=utot-udiv[:,np.newaxis,:,:]
vrot=vtot-vdiv[:,np.newaxis,:,:]

In [21]:
u1=u2rho_3d(np.squeeze(urot))
v1=v2rho_3d(np.squeeze(vrot))
ueast,vnorth=trans_vel_roms(u1,v1,angle)
ds3['u_rho'].values[:,0,:,:]=ueast
ds3['v_rho'].values[:,0,:,:]=vnorth
ds3['u'].values=urot
ds3['v'].values=vrot
ds3=ds3.drop_vars(['w','divof','Ro'])

In [22]:
ds3

<xarray.Dataset> Size: 4GB
Dimensions:     (depth: 1, eta_rho: 287, xi_rho: 287, time: 2148, xi_u: 286,
                 eta_v: 286)
Coordinates:
  * depth       (depth) float32 4B -2.0
    lat_rho     (eta_rho, xi_rho) float64 659kB ...
    lon_rho     (eta_rho, xi_rho) float64 659kB ...
  * time        (time) float64 17kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Dimensions without coordinates: eta_rho, xi_rho, xi_u, eta_v
Data variables:
    ocean_time  (time) float32 9kB ...
    u           (time, depth, eta_rho, xi_u) float32 705MB -0.0386 ... -0.1972
    u_rho       (time, depth, eta_rho, xi_rho) float64 1GB -0.03445 ... -0.2004
    v           (time, depth, eta_v, xi_rho) float32 705MB 0.02802 ... -0.09781
    v_rho       (time, depth, eta_rho, xi_rho) float64 1GB 0.03421 ... -0.06314
Attributes:
    CDI:          Climate Data Interface version 2.4.1 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon Oct 13 03:27:37 2025: ncks -x -v Th* nowavecase_modifie...
    CDO:          Climate Data Operators version 2.4.1 (https://mpimet.mpg.de...
    NCO:          netCDF Operators version 5.3.5 (Homepage = http://nco.sf.ne...

In [23]:
ds3.to_netcdf('/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/rot_helmholtz.0002.nc')
del ds3

# 500 m data

In [2]:
grid=xr.open_dataset('/meddy/simingzhang/Data/iceland_500m/sample_niskin_500m_grd.nc')
angle=np.tile(grid['angle'].values[np.newaxis,:,:],[1948,1,1])
angle.shape

(1948, 866, 866)

# HF

In [3]:
ds1=xr.open_dataset('/meddy/simingzhang/Data/iceland_500m/iceland_wave/afterspinup_z_his_depth.0002.nc')
u=ds1['u'].values
v=ds1['v'].values



In [4]:
ds1

<xarray.Dataset> Size: 23GB
Dimensions:     (time: 1948, depth: 1, eta_rho: 866, xi_u: 865, eta_v: 865,
                 xi_rho: 866)
Coordinates:
  * time        (time) float64 16kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
  * depth       (depth) float32 4B -2.0
Dimensions without coordinates: eta_rho, xi_u, eta_v, xi_rho
Data variables:
    ocean_time  (time) float32 8kB ...
    u           (time, depth, eta_rho, xi_u) float32 6GB 0.3315 ... -0.1182
    v           (time, depth, eta_v, xi_rho) float32 6GB -0.16 ... 0.1214
    w           (time, depth, eta_rho, xi_rho) float32 6GB ...
    AKv         (time, depth, eta_rho, xi_rho) float32 6GB ...
Attributes:
    CDI:                       Climate Data Interface version 2.4.1 (https://...
    Conventions:               CF-1.6
    history:                   Tue Oct 28 05:32:04 2025: cdo seltimestep,200/...
    nco_openmp_thread_number:  1
    nc_format:                 netCDF-4, zlib-compressed
    CDO:                       Climate Data Operators version 2.4.1 (https://...

In [7]:
u1=u2rho_3d(np.squeeze(u))
v1=v2rho_3d(np.squeeze(v))
ueast,vnorth=trans_vel_roms(u1,v1,angle)

In [10]:
n_time = ds1.dims['time']
n_depth = ds1.dims['depth'] 
n_eta_rho = ds1.dims['eta_rho']
n_xi_rho = ds1.dims['xi_rho']
n_time,n_depth,n_eta_rho,n_xi_rho

u_rho = xr.DataArray(
        ueast[:,np.newaxis,:,:],
        dims=['time', 'depth', 'eta_rho', 'xi_rho'],
        attrs={
            'long_name': 'Eastward velocity at RHO-points',
            'units': 'm/s',
            'coordinates': 'lon_rho lat_rho',
            'interpolation_method': 'linear_average'
        }
    )
    
v_rho = xr.DataArray(
    vnorth[:,np.newaxis,:,:], 
    dims=['time', 'depth', 'eta_rho', 'xi_rho'],
    attrs={
        'long_name': 'Northward velocity at RHO-points', 
        'units': 'm/s',
        'coordinates': 'lon_rho lat_rho',
        'interpolation_method': 'linear_average'
    }
)

# 添加到数据集
ds_new = ds1.assign({
    'u_rho': u_rho,
    'v_rho': v_rho
})


/tmp/ipykernel_3611271/333879331.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_time = ds1.dims['time']
/tmp/ipykernel_3611271/333879331.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_depth = ds1.dims['depth']
/tmp/ipykernel_3611271/333879331.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_eta_rho = ds1.dims['eta_rho']
/tmp/ipykernel_3611271/333879331.py:4: FutureWarning:

In [12]:
ds_new=ds_new.drop_vars(['AKv','w'])

In [13]:
ds_new.to_netcdf('/meddy/simingzhang/Data/iceland_500m/iceland_wave/wavecase_vel.nc')


In [15]:
del ds_new
del ds1

# smooth

In [ ]:
ds1=xr.open_dataset('/meddy/simingzhang/Data/iceland_500m/iceland_nowave/afterspinup_z_his_depth.0002.nc')
u=ds1['u'].values
v=ds1['v'].values
u1=u2rho_3d(np.squeeze(u))
v1=v2rho_3d(np.squeeze(v))
ueast,vnorth=trans_vel_roms(u1,v1,angle)

n_time = ds1.dims['time']
n_depth = ds1.dims['depth'] 
n_eta_rho = ds1.dims['eta_rho']
n_xi_rho = ds1.dims['xi_rho']
n_time,n_depth,n_eta_rho,n_xi_rho

u_rho = xr.DataArray(
        ueast[:,np.newaxis,:,:],
        dims=['time', 'depth', 'eta_rho', 'xi_rho'],
        attrs={
            'long_name': 'Eastward velocity at RHO-points',
            'units': 'm/s',
            'coordinates': 'lon_rho lat_rho',
            'interpolation_method': 'linear_average'
        }
    )
    
v_rho = xr.DataArray(
    vnorth[:,np.newaxis,:,:], 
    dims=['time', 'depth', 'eta_rho', 'xi_rho'],
    attrs={
        'long_name': 'Northward velocity at RHO-points', 
        'units': 'm/s',
        'coordinates': 'lon_rho lat_rho',
        'interpolation_method': 'linear_average'
    }
)

# 添加到数据集
ds_new = ds1.assign({
    'u_rho': u_rho,
    'v_rho': v_rho
})
ds_new.to_netcdf('/meddy/simingzhang/Data/iceland_500m/iceland_nowave/nowavecase_vel.nc')
del ds_new
del ds1